# Differential network analysis - network illustration

In [1]:
import pandas as pd

In [5]:
media_db = pd.read_csv("../../smetana/tests/data/media_db.tsv",sep="\t")

media_db

,medium,description,compound,name
0,LB,Lysogeny broth,adn,Adenosine
1,LB,Lysogeny broth,ala__L,L-Alanine
2,LB,Lysogeny broth,amp,AMP
3,LB,Lysogeny broth,arg__L,L-Arginine
4,LB,Lysogeny broth,aso3,Arsenite
...,...,...,...,...
80,M9,M9 minimal medium,ni2,Ni2+
81,M9,M9 minimal medium,o2,O2
82,M9,M9 minimal medium,pi,Phosphate
83,M9,M9 minimal medium,so4,Sulfate


In [ ]:

don_rec_df_unstacked = don_rec_df.unstack().fillna(0)

diff_df = (don_rec_df.xs("EL-")-don_rec_df.xs("NONE")).fillna(0)


index_B = diff_df.map(abs).sum()>diff_df.map(abs).sum().mean()
index_B = index_B[index_B].index.values
index_A = diff_df.map(abs).T.sum()>diff_df.map(abs).T.sum().mean()
index_A = index_A[index_A].index.values

index = list(set(list(index_B)+list(index_A)))
index.sort()

diff_df = diff_df.loc[index,index]
diff_df = diff_df.stack()

In [147]:
import matplotlib.pyplot as plt
import matplotlib as mpl

def color_from_bwr(val, vmin=-1, vmax=1):
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    cmap = mpl.cm.bwr
    return mpl.colors.to_hex(cmap(norm(val)))

In [148]:
from pyvis.network import Network

In [161]:
network_data = diff_df.copy()

net = Network(directed=True)
nodes = list(set(list(network_data.index.get_level_values(0).unique())+ list(network_data.index.get_level_values(1).unique())))
nodes_mets = [node for node in nodes if node.startswith("M_")]
nodes_members = list(set(nodes)-set(nodes_mets))

net.add_nodes(nodes_mets,color=["black" for node in nodes_mets],value=[1 for node in nodes_mets])
net.add_nodes(nodes_members)
maximum_overlap = network_data.map(abs).max()
for index,value in network_data.items():

    net.add_edge(index[0],index[1],width=abs(value)*100,color=color_from_bwr(value/maximum_overlap)) 

net.barnes_hut(gravity=-500,damping=0.09)

#gravity=-100, central_gravity=0.1, spring_length=150, spring_strength=0.0015, damping=0.09, overlap=0.8
template = net.templateEnv.get_template(net.path)
net.template = template

net.show("network_file.html")

network_file.html
